In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from pathlib import Path
import os
from dotenv import load_dotenv


e:\genai\gen\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
def load_data(document):
    context=[]
    document_path=Path(document)
    files=list(document_path.glob("**/*.pdf"))
    for file in files:
        try:
            doc=PyPDFLoader(str(file))
            print(f"loadintg file : {file}")
            d=doc.load()
            context.extend(d)
        except Exception as e:
            print(f"cannot load file : {file} due to {e}")
    return context

In [4]:
documents=load_data(r"D:\college\Sem 6\computer_network")

loadintg file : D:\college\Sem 6\computer_network\BCAC603 (NETWORKING) NOTES.pdf
loadintg file : D:\college\Sem 6\computer_network\COMPUTER NETWORKS NOTES.pdf
loadintg file : D:\college\Sem 6\computer_network\Computer Networks organizer.pdf


In [5]:
documents[0].page_content

'1 | P a g e \n \nMODULE-I \n1. Definition of Computer Networking  \nComputer networking is the process of connecting two or more computers and \ndigital devices so they can communicate, share data, exchange information, \nand use common resources like printers, storage, and the internet. These wired \nor wireless connections help devices work together smoothly, making \ncommunication faster, improving data sharing, reducing costs, and supporting \nbetter management of information and operations. \n \n2. Importance of Networking  \n1. Enables Fast Communication \nNetworking allows quick sharing of messages, files, and information, \nmaking teamwork smoother and faster. \n2. Supports Resource Sharing \nDevices such as printers, scanners, and storage can be shared, reducing \nthe need for separate equipment. \n3. Makes Data Management Easier \nImportant data can be stored and controlled from one central place, \nmaking it simple to update and protect. \n4. Improves Productivity \nBecause

In [6]:
def text_splitter(documents,chunk_size=1000,chunk_overlap=30):
    splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n","."," "]
    )
    new_document=splitter.split_documents(documents)
    return new_document

In [7]:
after_split=text_splitter(documents)

In [8]:
after_split[0].page_content

'1 | P a g e \n \nMODULE-I \n1. Definition of Computer Networking  \nComputer networking is the process of connecting two or more computers and \ndigital devices so they can communicate, share data, exchange information, \nand use common resources like printers, storage, and the internet. These wired \nor wireless connections help devices work together smoothly, making \ncommunication faster, improving data sharing, reducing costs, and supporting \nbetter management of information and operations. \n \n2. Importance of Networking  \n1. Enables Fast Communication \nNetworking allows quick sharing of messages, files, and information, \nmaking teamwork smoother and faster. \n2. Supports Resource Sharing \nDevices such as printers, scanners, and storage can be shared, reducing \nthe need for separate equipment. \n3. Makes Data Management Easier \nImportant data can be stored and controlled from one central place, \nmaking it simple to update and protect. \n4. Improves Productivity'

In [9]:
db="chroma"
model=HuggingFaceEndpointEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
vector_space=Chroma.from_documents(
    documents=after_split,
    embedding=model,
    persist_directory=db,
)

In [10]:
vect=vector_space.as_retriever()
ans=vect.invoke("what is ipv4")
for i in ans:
    print(i.metadata["source"],i.metadata["page"])

D:\college\Sem 6\computer_network\BCAC603 (NETWORKING) NOTES.pdf 247
D:\college\Sem 6\computer_network\BCAC603 (NETWORKING) NOTES.pdf 247
D:\college\Sem 6\computer_network\BCAC603 (NETWORKING) NOTES.pdf 251
D:\college\Sem 6\computer_network\BCAC603 (NETWORKING) NOTES.pdf 251


In [15]:
def rag_pipeline(question):
    vect=vector_space.as_retriever()
    cont=vect.invoke(question)
    context="\n\n".join([doc.page_content for doc in cont])if cont else "no context is present in this document"
    chat_model=init_chat_model("groq:openai/gpt-oss-120b")
    prompt=PromptTemplate(
        input_variables=["context","question"],
        template="Use ONLY the following context:\n{context}\n\nIf the answer is not in the context, say 'Not found in document.'\n\nQuestion: {question}\nAnswer:")
    chain=prompt | chat_model | StrOutputParser()
    return chain.invoke({"context":context,"question":question})

In [19]:
rag_pipeline("tcp")

'TCP is a connection‑oriented, reliable transport protocol that creates a virtual connection between two TCP endpoints to send data. It adds connection‑oriented and reliability features to IP and uses flow‑control and error‑control mechanisms at the transport level.'

In [13]:
# from langchain.chat_models import init_chat_model
# from langchain_core.prompts import PromptTemplate
# from langchain_core.output_parsers import StrOutputParser
# from langchain_chroma import Chroma
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_community.document_loaders import PyPDFLoader
# from langchain_huggingface import HuggingFaceEmbeddings
# from pathlib import Path
# import os
# from dotenv import load_dotenv

# load_dotenv()

# def load_data(document):
#     context = []
#     document_path = Path(document)
#     files = list(document_path.glob("**/*.pdf"))
#     for file in files:
#         try:
#             doc = PyPDFLoader(str(file))
#             print(f"Loading file: {file}")
#             d = doc.load()
#             context.extend(d)
#         except Exception as e:
#             print(f"Cannot load file: {file} due to {e}")
#     return context

# documents = load_data(r"D:\college\Sem 6\computer_network")

# def text_splitter(documents, chunk_size=1000, chunk_overlap=30):
#     splitter = RecursiveCharacterTextSplitter(
#         chunk_size=chunk_size,
#         chunk_overlap=chunk_overlap,
#         length_function=len,
#         separators=["\n\n", "\n", ".", " "]
#     )
#     return splitter.split_documents(documents)

# after_split = text_splitter(documents)

# db = "chroma_db"
# embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# vector_space = Chroma.from_documents(
#     documents=after_split,
#     embedding=embedding,
#     persist_directory=db,
# )


# def rag_pipeline(question):
#     retriever = vector_space.as_retriever()
#     docs = retriever.invoke(question)
#     context = "\n\n".join([doc.page_content for doc in docs]) if docs else "Not found in document."
    
#     if context == "Not found in document.":
#         return context
    
#     chat_model = init_chat_model("groq:openai/gpt-oss-120b")
#     prompt = PromptTemplate(
#         input_variables=["context", "question"],
#         template="Use ONLY the following context:\n{context}\n\n"
#                  "If the answer is not in the context, say 'Not found in document.'\n\n"
#                  "Question: {question}\nAnswer:"
#     )
#     chain = prompt | chat_model | StrOutputParser()
#     return chain.invoke({"context": context, "question": question})

# print(rag_pipeline("TCP"))


In [14]:
rag_pipeline("what is dsa")

AIMessage(content='Not found in document.', additional_kwargs={'reasoning_content': 'The user asks: "what is dsa". Need to answer using only the provided context. The context includes information about SMTP, MUA, MSA, DNSSEC, reverse DNS. No mention of DSA. So answer: Not found in document.'}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 984, 'total_tokens': 1051, 'completion_time': 0.142550953, 'completion_tokens_details': {'reasoning_tokens': 53}, 'prompt_time': 0.041740375, 'prompt_tokens_details': None, 'queue_time': 0.405879743, 'total_time': 0.184291328}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e5b4e54fbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a09693-1541-76d0-961d-e15944102a0e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 984, 'output_tokens': 67, 'total_tokens': 1051, 'output_token_details': {'reasoning': 53}})